# Core Analysis Objective

Understand how shoppers progress from viewing products to adding to cart and purchasing, pinpoint where the biggest drop-offs happen, and see how conversion varies across product categories, brands, and price tiers.

Specifically, we can frame this around 4 concrete questions:

**Where do shoppers leak?** What percentage of sessions/users move from view → cart → purchase, and where is the steepest drop-off?

**How do journeys behave in practice?** Do events always happen in view → cart → purchase order within a session, or do users buy without a recorded cart event, repeat views/carts, or span multiple sessions?

**What drives higher or lower conversion?** Which category_code and brand segments convert best vs. struggle most?

**How does price affect drop-off?** Do higher-priced items have steeper cart-to-purchase drop-offs than low- or mid-priced items?

In [6]:
# Importing Libraries

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

import pyarrow as pa
import pyarrow.parquet as pq

import kagglehub
from kagglehub import KaggleDatasetAdapter
import zlib

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [7]:
# Listing all the files under the input folder

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv
/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv


# Loading the Data 

We'll be initially loading 500,000 records rather that loading the entire dataset (which has almost 42M records). 

Rather than randomly sampling individual records, sampling by user_id or user_session would allow me to preserve the relationships between events and understand the complete customer journey: 

**View → Add to Cart → Purchase.**

Since we have 9.24 million unique user session IDs, if we take 5% of that, it would give us approximately 462,000 sessions. 
Reading 9.24 million user session IDs and randomly sampling 5% of that would waste a lot of time and RAM.

The workaround? -> Hashing the session IDs

The same session ID will always produce the same hash value.

*So, how do we select the 5%?* 
We use the below rule.

```{python}
hash_value % 20 == 0
```

This means:
- Calculate the hash value of a session ID.
- Divide it by 20.
- If the remainder is 0, keep that session.
- Otherwise, discard it.

*Why 20?* Because approximately 1 out of every 20 hash values will satisfy this condition.

**That's approximately 5%!**


In [8]:
file_path = r"/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv"

cols_to_use = ["user_id",
               "event_time",
               "event_type", 
               "brand", 
               "price",
               "category_code", 
               "user_session",
               "product_id"]
chunksize = 500_000
# rng = np.random.default_rng(42)  # reproducible random sampling

sample_rate = 0.05  # e.g., 5% of sessions
threshold = int(sample_rate * (2**32))

In [9]:
def in_session_sample(session_series: pd.Series) -> pd.Series:
    # Initial check to see whether the sample rate is between 0 and 1
    if sample_rate <= 0:
        return pd.Series(False, index=session_series.index)
    if sample_rate >= 1:
        return session_series.notna()

    hashes = session_series.fillna("").map( # Replaces missing session IDs with empty strings.
        lambda s: zlib.crc32(s.encode("utf-8")) # Converts the session ID into a numerical hash.
        if s else 2**32
    )
    return hashes < threshold

# def in_session_sample(session_series: pd.Series) -> pd.Series:
#     hashes = session_series.fillna("").map( # Replaces missing session IDs with empty strings.
#         lambda s: zlib.crc32(s.encode("utf-8")) # Converts the session ID into a numerical hash.
#         if s else 2**32 
#     )
#     return hashes % 20 == 0 #Checks whether the hash is divisible by 20.


reader = pd.read_csv(file_path, usecols=cols_to_use, chunksize=chunksize)
writer = None


# For every chunk:
# 1. Check whether the event is view, cart, or purchase.
# 2. Check whether its session ID passes the 5% hash rule.
# 3. Keep the row only if both conditions are satisfied.

for chunk in reader:
    mask = (
        chunk["event_type"].isin(["view", "cart", "purchase"]) # core events in the funnel cycle
        & in_session_sample(chunk["user_session"]) # only focussing on the user session column for now
    )
    chunk = chunk[mask].copy()

    if chunk.empty:
        continue

    # Parse the timestamps and assigning appropriate data types to columns
    chunk["event_time"] = pd.to_datetime(chunk["event_time"], errors="coerce") 
    chunk["event_type"] = chunk["event_type"].astype("category")
    if chunk["price"].dtype == "float64":
        chunk["price"] = chunk["price"].astype("float32")

    # Converts the current Pandas chunk into an Apache Arrow Table
    # (the in-memory columnar format PyArrow uses to write Parquet files)
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter("Oct_500K_sample.parquet", table.schema, compression="snappy")
    writer.write_table(table)

if writer is not None:
    writer.close()


# Data Cleaning and Preprocessing

## Understanding the Data

In [8]:
cleaned_data_path = r"/kaggle/input/notebooks/gojooooo/funnel-analysis-for-ecommerce-store/Oct_500K_sample.parquet"

df = pd.read_parquet(cleaned_data_path)
df.head()

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,view,44600062,None,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:15+00:00,view,44600062,None,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
2,2019-10-01 00:00:44+00:00,view,26500313,None,lucente,267.190002,525856698,f72ea16b-4ec3-44f8-8fcd-35d89984b744
3,2019-10-01 00:01:12+00:00,view,26500144,None,lucente,259.209991,525856698,f72ea16b-4ec3-44f8-8fcd-35d89984b744
4,2019-10-01 00:01:31+00:00,view,26200210,None,None,98.839996,555447493,8c3e1835-2196-4022-926c-7f0da0ae0aa5


In [11]:
df.shape

(2123581, 8)

In [12]:
df.head()

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,view,44600062,None,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:15+00:00,view,44600062,None,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
2,2019-10-01 00:00:44+00:00,view,26500313,None,lucente,267.190002,525856698,f72ea16b-4ec3-44f8-8fcd-35d89984b744
3,2019-10-01 00:01:12+00:00,view,26500144,None,lucente,259.209991,525856698,f72ea16b-4ec3-44f8-8fcd-35d89984b744
4,2019-10-01 00:01:31+00:00,view,26200210,None,None,98.839996,555447493,8c3e1835-2196-4022-926c-7f0da0ae0aa5


In [14]:
df.tail()

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
2123576,2019-10-31 23:59:27+00:00,view,1480156,computers.desktop,neo,622.669983,543124499,58d5c559-5358-48bb-9f25-9a0d43297f9b
2123577,2019-10-31 23:59:29+00:00,view,4804695,electronics.audio.headphone,fender,279.290009,566280399,63dd0460-2ee7-4df3-a0b0-17d78f3029a2
2123578,2019-10-31 23:59:47+00:00,view,1004503,electronics.smartphone,oppo,514.559998,523341345,97b84b1e-b898-4189-ba88-7c28d0f6f919
2123579,2019-10-31 23:59:55+00:00,view,1480709,computers.desktop,pulser,592.010010,543124499,58d5c559-5358-48bb-9f25-9a0d43297f9b
2123580,2019-10-31 23:59:59+00:00,view,13300120,None,swisshome,155.729996,525266378,6e57d2d7-6022-46e6-81d6-fa77f14cefd8


In [15]:
print("\n--- Data Types ---")
print(df.dtypes)


--- Data Types ---
event_time       datetime64[ns, UTC]
event_type                  category
product_id                     int64
category_code                 object
brand                         object
price                        float32
user_id                        int64
user_session                  object
dtype: object


In [16]:
print("\n--- Missing Values (%) ---")
print((df.isna().mean() * 100).round(2))


--- Missing Values (%) ---
event_time        0.00
event_type        0.00
product_id        0.00
category_code    31.91
brand            14.47
price             0.00
user_id           0.00
user_session      0.00
dtype: float64


In [17]:
print("\n--- Time Range ---")
print("Min event_time:", df["event_time"].min())
print("Max event_time:", df["event_time"].max())


--- Time Range ---
Min event_time: 2019-10-01 00:00:00+00:00
Max event_time: 2019-10-31 23:59:59+00:00


In [18]:
print("\n--- Event Type Counts ---")
print((df["event_type"].value_counts(dropna=False) * 100.0/ df["event_type"].shape[0]).round(2))


--- Event Type Counts ---
event_type
view        96.07
cart         2.18
purchase     1.74
Name: count, dtype: float64


In [19]:
print("\n--- Price Summary & Anomalies ---")
print(df["price"].describe())
print("Price <= 0 count:", (df["price"] <= 0).sum())


--- Price Summary & Anomalies ---
count    2.123581e+06
mean     2.906560e+02
std      3.583877e+02
min      0.000000e+00
25%      6.641000e+01
50%      1.629400e+02
75%      3.585700e+02
max      2.574070e+03
Name: price, dtype: float64
Price <= 0 count: 3518


In [20]:
print("\n--- Duplicate & Session/User Sanity ---")
print("Exact duplicate rows:", df.duplicated().sum())

# Check if a single user_session ever maps to multiple user_ids
sessions_per_user = df.groupby("user_session", observed=True)["user_id"].nunique()
print("Sessions with >1 user_id:", (sessions_per_user > 1).sum())


--- Duplicate & Session/User Sanity ---
Exact duplicate rows: 1477
Sessions with >1 user_id: 12


OKAYY so here is what we understand from this:
1. Missing values - Category code (32%) and brand (14%)
2. Event distribution - 96% views, 2% carts, 1.7% purchases
3. Price column - Mean 290.66, median 162.94 (possible right skewness)
4. There are 3518 records with Price = 0
5. 12 sessions associated with multiple user IDs

Before we investigate these, let's first create features out of the date column. We'll be extracting the year, month, and the day, along with the time zone, from the date column. 

In [9]:
# df['Year'] = df['event_time'].dt.year
# df['Month'] = df['event_time'].dt.month
df['Day'] = df['event_time'].dt.day
df['Day_of_week'] = df['event_time'].dt.dayofweek # 0 = Monday, 6 = Sunday
df['Day_of_week_name'] = df['event_time'].dt.day_name() # 'Monday', 'Tuesday', etc.

df['Time'] = df['event_time'].dt.time

In [43]:
df['event_time'].dt.tz

<UTC>

Another thing that I'm going to do is focused on the category code. We can see that there are different sections within the category code, which are separated by the '.' delimiter. We can extract the main category and the leaf category from this feature. 

In [10]:
df['Main_category'] = df['category_code'].str.split('.').str[0]
df["Leaf_category"] = df['category_code'].str.split('.').str[-1]

In [12]:
df.tail(3)

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session,Day,Day_of_week,Day_of_week_name,Time,Main_category,Leaf_category
2123578,2019-10-31 23:59:47+00:00,view,1004503,electronics.smartphone,oppo,514.559998,523341345,97b84b1e-b898-4189-ba88-7c28d0f6f919,31,3,Thursday,23:59:47,electronics,smartphone
2123579,2019-10-31 23:59:55+00:00,view,1480709,computers.desktop,pulser,592.010010,543124499,58d5c559-5358-48bb-9f25-9a0d43297f9b,31,3,Thursday,23:59:55,computers,desktop
2123580,2019-10-31 23:59:59+00:00,view,13300120,None,swisshome,155.729996,525266378,6e57d2d7-6022-46e6-81d6-fa77f14cefd8,31,3,Thursday,23:59:59,None,None


## Data Investigation

### Duplicate records (Multiple user within the same user session).

In [47]:
df[df.groupby('user_session')['user_id']\
     .transform('nunique') > 1]['Day'].unique().tolist()

[3, 5, 7, 8, 19, 20, 22, 24, 27]

In [50]:
df[df.groupby('user_session')['user_id']\
     .transform('nunique') > 1]['Day_of_week_name'].unique().tolist()

['Thursday', 'Saturday', 'Monday', 'Tuesday', 'Sunday']

In [13]:
df[df.groupby('user_session')['user_id']\
     .transform('nunique') > 1]['Main_category'].value_counts()

Main_category
electronics    64
computers       2
Name: count, dtype: int64

In [14]:
df[df.groupby('user_session')['user_id']\
     .transform('nunique') > 1]['Leaf_category'].value_counts()

Leaf_category
smartphone    59
clocks         5
cpu            2
Name: count, dtype: int64

In [16]:
# Identify sessions associated with multiple users
session_user_counts = (
    df.groupby('user_session')['user_id']
    .nunique()
)

multi_user_sessions = session_user_counts[
    session_user_counts > 1
]

print("Number of affected sessions:", len(multi_user_sessions))
print("Number of affected records:",
      df[df['user_session'].isin(multi_user_sessions.index)].shape[0])

Number of affected sessions: 12
Number of affected records: 74


In [17]:
affected_df = df[
    df['user_session'].isin(multi_user_sessions.index)
].sort_values(['user_session', 'event_time'])

In [23]:
sub_aff = affected_df[['event_time','Time', 'Main_category', 'Leaf_category', 'brand', 'user_id', 'user_session']]

In [24]:
sub_aff.head(30)

,event_time,Time,Main_category,Leaf_category,brand,user_id,user_session
409200,2019-10-07 11:39:18+00:00,11:39:18,electronics,smartphone,xiaomi,557695402,4e014d1d-b326-4e20-8c67-ec2c5616121b
409203,2019-10-07 11:39:23+00:00,11:39:23,electronics,smartphone,xiaomi,557695402,4e014d1d-b326-4e20-8c67-ec2c5616121b
409270,2019-10-07 11:40:37+00:00,11:40:37,electronics,smartphone,xiaomi,557695402,4e014d1d-b326-4e20-8c67-ec2c5616121b
409520,2019-10-07 11:45:33+00:00,11:45:33,electronics,smartphone,xiaomi,524545429,4e014d1d-b326-4e20-8c67-ec2c5616121b
409542,2019-10-07 11:46:08+00:00,11:46:08,electronics,smartphone,xiaomi,524545429,4e014d1d-b326-4e20-8c67-ec2c5616121b
409604,2019-10-07 11:47:22+00:00,11:47:22,electronics,smartphone,xiaomi,524545429,4e014d1d-b326-4e20-8c67-ec2c5616121b
1251929,2019-10-19 05:34:48+00:00,05:34:48,electronics,clocks,casio,561783568,5c10f563-8741-4e9c-8aaf-9231f6dfbcc0
1252069,2019-10-19 05:36:39+00:00,05:36:39,electronics,clocks,casio,561783575,5c10f563-8741-4e9c-8aaf-9231f6dfbcc0
1345250,2019-10-20 09:57:25+00:00,09:57:25,electronics,smartphone,samsung,562195447,8b931108-97be-4997-992a-b2763783e813
1345275,2019-10-20 09:57:54+00:00,09:57:54,electronics,smartphone,samsung,562195445,8b931108-97be-4997-992a-b2763783e813


In [37]:
affected_df.groupby(['category_code', 'event_type'])\
.size()\
.sort_index(level=['category_code', 'event_type'], ascending=[False, False])

/tmp/ipykernel_50/678241804.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  affected_df.groupby(['category_code', 'event_type'])\


category_code             event_type
electronics.smartphone    view          59
                          purchase       0
                          cart           0
electronics.clocks        view           5
                          purchase       0
                          cart           0
computers.components.cpu  view           2
                          purchase       0
                          cart           0
dtype: int64

In [35]:
# Analyzing records of those with missing categories
affected_df[affected_df['category_code'].isna()]\
.groupby(['user_session', 'event_type'])\
.size()\
.sort_index(level=['user_session', 'event_type'], ascending=[True, False])

/tmp/ipykernel_50/1430172434.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['user_session', 'event_type'])\


user_session                          event_type
a5fae733-b1a3-4625-b151-dcecef42f14e  view          3
                                      purchase      0
                                      cart          0
ccea5076-fff5-49bf-ad1c-cbc3f441afaf  view          2
                                      purchase      0
                                      cart          0
d11e8168-e9f7-4d4c-883b-c4cb4a020a62  view          2
                                      purchase      1
                                      cart          0
dtype: int64

In [39]:
df[df['user_session'] == 'd11e8168-e9f7-4d4c-883b-c4cb4a020a62']

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session,Day,Day_of_week,Day_of_week_name,Time,Main_category,Leaf_category
1663008,2019-10-24 19:15:34+00:00,view,22700528,None,force,97.809998,563816594,d11e8168-e9f7-4d4c-883b-c4cb4a020a62,24,3,Thursday,19:15:34,None,None
1663162,2019-10-24 19:18:43+00:00,view,22700528,None,force,97.809998,563816593,d11e8168-e9f7-4d4c-883b-c4cb4a020a62,24,3,Thursday,19:18:43,None,None
1663279,2019-10-24 19:22:07+00:00,purchase,22700528,None,force,97.809998,563816593,d11e8168-e9f7-4d4c-883b-c4cb4a020a62,24,3,Thursday,19:22:07,None,None


In [42]:
df['multi_user_session'] = (
    df.groupby('user_session')['user_id']
      .transform('nunique') > 1
)

So this is what we can document regarding our investigation on the multi user sessions:

- The investigation identified `74 records across 12 sessions` where multiple user_id values were associated with the same user_session.
- The user ID transitions consistently occurred during product `view` event type.
- The transitions appeared relatively isolated, with users generally generating consecutive events rather than repeatedly alternating within a session.
- The underlying cause remains unknown. It could be a tracking inconsistency, session attribution issue or another behaviour in the original data collection process. We don't have enough information to determine which.

Since it wont be causing a major issue in our analysis, we will be retaining the records. To identify these records, we have introduced a flag named `multi_user_session`.

**NOTE: just for info, there are more # of distinct sessions compared to user_ids.**

### Examine records with zero prices.

In [ ]:
### Understand missing category and brand values.